(course-core-02)=

# Module 2: Native Forms

**Welcome back, Apprentice Master.** In Module 1, you discovered that MolSysMT treats every external file format or third-party object as a **Form**.

While MolSysMT is form-agnostic, it is not homeless. When you want maximum computational speed, zero-copy structural queries, and complete control over your data, you rely on **Native Forms**.

At the center of this ecosystem is **`molsysmt.MolSys`**, a modular orchestrator that unifies domain-specific native objects: **`Topology`** (covalent graph & hierarchy), **`Structures`** (coordinates, boxes & trajectories), and **`MolecularMechanics`** (forcefield parameters, charges & energies).

```{admonition} Glossary: Native Form
:class: dropdown info
A **Native Form** is any data structure built directly within MolSysMT (`molsysmt.MolSys`, `molsysmt.Topology`, `molsysmt.Structures`, `molsysmt.TopologyDict`, `file:h5msm`) designed for optimal memory layout and maximum analytical performance. See the {ref}`user-foundations` guide for a full list of native forms.
```

(course-core-02-learning-outcomes)=
> **🎯 Learning Outcomes**
>
> By the end of this module, you will be able to:
> - Instantiate and work with the **`molsysmt.MolSys`** native orchestrator object.
> - Understand the modular architecture of component objects (`Topology`, `Structures`, `MolecularMechanics`).
> - Leverage disk-based **H5MSM** containers for high-performance storage.
> - Access lightweight dictionary forms (`TopologyDict`, `StructuresDict`, `MolSysDict`) for zero-overhead Python data manipulation.

### 1. The Native Orchestrator: `molsysmt.MolSys`

Let's begin by importing MolSysMT, loading a molecular system, and converting it to MolSysMT's primary in-memory native object: **`molsysmt.MolSys`**.

In [1]:
import molsysmt as msm
from molsysmt import systems

# Load T4 Lysozyme and convert it into the native MolSys object
lysozyme_file = systems['T4 lysozyme L99A']['181l.bcif.gz']
molsys = msm.convert(lysozyme_file, to_form='molsysmt.MolSys')

When inspecting `molsys`, notice the difference between Python's internal class implementation (`type()`) and MolSysMT's canonical form string (`msm.get_form()`):

In [2]:
print(f"Python class type : {type(molsys)}")
print(f"MolSysMT form name: {msm.get_form(molsys)}")

Python class type : <class 'molsysmt.native.molsys.MolSys'>
MolSysMT form name: molsysmt.MolSys


:::{hint}
:class: dropdown
**molsysmt.MolSys**: The primary in-memory native container of MolSysMT. It orchestrates modular component objects (`topology`, `structures`, `molecular_mechanics`, and expanding domain modules). See API doc: {func}`molsysmt.native.MolSys`.
:::

### 2. Modular Domain Components: `Topology`, `Structures`, and `MolecularMechanics` 

Rather than storing all data in a single monolithic array, `molsysmt.MolSys` delegates domain responsibilities to specialized native component objects:
- **`molsys.topology` (`molsysmt.Topology`)**: Manages the covalent graph, atom names, element types, groups/residues, components, chains, molecules, entities, and chemical bonds. *(Explored in depth in [Module 05: Molecular Anatomy](../00_Common_Core/index.md))*
- **`molsys.structures` (`molsysmt.Structures`)**: Manages spatial coordinates array `(n_structures, n_atoms, 3)`, periodic box vectors `(n_structures, 3, 3)`, time steps, velocities, and frame observables. *(Explored in depth in [Module 21: Data Analyst](../00_Common_Core/index.md) & [Module 40: The Physics Lab](../00_Common_Core/index.md))*
- **`molsys.molecular_mechanics` (`molsysmt.MolecularMechanics`)**: Manages forcefield definitions, formal charges, partial charges, non-bonded parameters, and potential energy.

Let's inspect the form of each component attribute inside our `molsys` instance:

In [3]:
print(f"Topology component form: {msm.get_form(molsys.topology)}")
print(f"Structures component form: {msm.get_form(molsys.structures)}")
print(f"Molecular Mechanics component form: {msm.get_form(molsys.molecular_mechanics)}")

Topology component form: molsysmt.Topology
Structures component form: molsysmt.Structures
Molecular Mechanics component form: molsysmt.MolecularMechanics


:::{hint}
:class: dropdown
**molsysmt.Topology**: Native object representing the chemical and structural hierarchy (atoms, groups, chains, bonds). See API docs for native classes: {func}`molsysmt.native.Topology`, {func}`molsysmt.native.Structures`, and {func}`molsysmt.native.MolecularMechanics`.
:::

### 3. High-Performance Disk Storage: H5MSM Files (`file:h5msm`)

The **H5MSM** format (`file:h5msm`) is the disk-based counterpart of `molsysmt.MolSys`. Built on top of HDF5, it stores system topology, trajectory coordinates, box vectors, and physical observables in binary form with maximum I/O performance.

Unlike third-party formats, **H5MSM** allows chunked execution and partial trajectory loading without materializing the full dataset into memory. *(Trajectory I/O workflows are covered in [Module 14: The Virtual Lab](../00_Common_Core/index.md) & [Module 47: Pipeline Developer](../00_Common_Core/index.md))*.

In [4]:
# Inspect an H5MSM file from the demo repository
h5msm_file = systems['T4 lysozyme L99A']['181l.h5msm']
print(f"Form of the file: {msm.get_form(h5msm_file)}")

Form of the file: file:h5msm


:::{hint}
:class: dropdown
**file:h5msm**: Native disk-based HDF5 container for MolSysMT systems, supporting complete trajectory datasets and structural states. See User Guide: {ref}`user-foundations`.
:::

### 4. Lightweight Native Dictionaries: `TopologyDict`, `StructuresDict`, and `MolSysDict` 

For high-speed Python pipelines, API serialization, or low-latency inspection, MolSysMT provides dictionary-based native forms:
- **`molsysmt.TopologyDict`**: Lightweight dictionary containing pure topological arrays and connectivity maps.
- **`molsysmt.StructuresDict`**: Lightweight dictionary containing raw coordinate arrays and frame observables.
- **`molsysmt.MolSysDict`**: Dictionary representation bundling topology, structures, and mechanics data.

Because these forms use pure Python dictionaries, all internal data is transparently accessible via standard key lookup. Let's convert `molsys.topology` into a `TopologyDict` and explore its data:

In [5]:
# Convert topology component to a native TopologyDict
topo_dict = msm.convert(molsys.topology, to_form='molsysmt.TopologyDict')

print(f"Form found: {msm.get_form(topo_dict)}")
print(f"Dictionary keys: {list(topo_dict.data.keys())}")
print(f"\nNumber of atom records: {len(topo_dict.data['atoms'])}")
print(f"Number of group records: {len(topo_dict.data['groups'])}")

Form found: molsysmt.TopologyDict
Dictionary keys: ['format', 'kind', 'version', 'metadata', 'atoms', 'groups', 'bonds', 'chains', 'molecules', 'entities']

Number of atom records: 1441
Number of group records: 302


:::{hint}
:class: dropdown
**molsysmt.TopologyDict**: Lightweight native Python dictionary form holding topology attributes for zero-overhead iteration and serialization. See API doc: {func}`molsysmt.native.TopologyDict`.
:::

--- 

### 🏆 Challenge 2: The Native Architect

1. Load the **SARS-CoV-2 Protease** using its PDB ID: `'pdb_id:6LU7'`.
2. Convert it into a native **`molsysmt.MolSys`** object.
3. Use `msm.get_form()` on its `topology` and `structures` attributes to verify their native forms.
4. Convert the native object into a **`molsysmt.TopologyDict`** and inspect its `'atoms'` key length.

Mastering native forms gives you maximum performance and flexibility. In **Module 3**, we will learn how to compose systems using multiple sources.

```{key-takeaway}
MolSysMT native objects (`MolSys`, `Topology`, `Structures`, `H5MSM`, and `Dict` forms) provide a modular, high-performance architecture for structural biology.
```

(course-core-02-see-also)=
:::{seealso}
:class: dropdown
**API Documentation for Native Forms in this Module:**
- {func}`molsysmt.native.MolSys` — Primary native in-memory orchestrator container.
- {func}`molsysmt.native.Topology` — Native topology hierarchy and covalent graph object.
- {func}`molsysmt.native.Structures` — Native structural coordinate and trajectory object.
- {func}`molsysmt.native.MolecularMechanics` — Native molecular mechanics and forcefield object.
- {func}`molsysmt.native.TopologyDict` — Lightweight topology dictionary form.
- {func}`molsysmt.native.StructuresDict` — Lightweight structural dictionary form.
- {func}`molsysmt.native.MolSysDict` — Lightweight complete system dictionary form.

**Related Course Modules & Guides:**
- Previous Module: [Module 1: The Form-Agnostic Philosophy](../00_Common_Core/01_The_Form_Agnostic_Philosophy.ipynb)
- Next Module: [Module 3: The Multi-Source Assembler](../00_Common_Core/03_The_Multi_Source_Assembler.ipynb)
- User Guide: {ref}`user-foundations`
:::